> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 2 · Notebook 06 — Volatility modelling and forecasting

**Sessions:** S6 (Volatility modelling & forecasting) · Clinic W2 · [Lesson plan](../../docs/lessons/PART_02_QUANT_TOOLKIT.md)

**You will:**
1. Build an EWMA volatility estimate.
2. Fit GARCH-t and GJR-GARCH models.
3. Run an honest out-of-sample volatility forecast competition with QLIKE.

How these notebooks work: the loading and plotting code is written for you. Cells marked **✍️ Your turn** need 1–5 lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p2lib.py is in notebooks/part02/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p2lib as p

p.use_course_style()
pd.set_option("display.float_format", "{:,.4f}".format)
prices = p.load_prices()          # dates × 10 tickers (course data via P2_DATA, else synthetic)
rets = p.log_returns(prices)      # daily log returns
prices.tail(3)

## 1. EWMA (RiskMetrics)

$\sigma^2_t = \lambda\,\sigma^2_{t-1} + (1-\lambda)\,r^2_{t-1}$ with $\lambda = 0.94$

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
r = rets["SPY"].to_numpy()

def ewma_next(prev_var, prev_ret, lam=0.94):
    return lam * prev_var + (1 - lam) * prev_ret ** 2

var = np.empty(len(r)); var[0] = r[:20].var()
for t in range(1, len(r)):
    nxt = ewma_next(var[t - 1], r[t - 1])
    if nxt is Ellipsis:
        break                                        # not done yet
    var[t] = nxt
lam = 0.94
var = p.check("EWMA variance", var, p.ewma_var(r, lam))

In [ ]:
vol = pd.DataFrame({"EWMA (λ = 0.94)": np.sqrt(var * 252),
                    "21-day rolling": rets["SPY"].rolling(21).std().shift(1) * np.sqrt(252)}, index=rets.index)
ax = (vol * 100).plot(title="SPY volatility estimates (% annualized)"); ax.set_xlabel(""); plt.show()

## 2. GARCH-t and GJR-GARCH (leverage effect)

In [ ]:
from arch import arch_model
y = 100 * rets["SPY"]                               # percent returns keep the optimizer stable
garch = arch_model(y, vol="GARCH", p=1, q=1, dist="t").fit(disp="off")
gjr = arch_model(y, vol="GARCH", p=1, o=1, q=1, dist="t").fit(disp="off")
pars = garch.params
persistence = pars["alpha[1]"] + pars["beta[1]"]
long_run = np.sqrt(pars["omega"] / (1 - persistence) * 252) / 100
print(f"GARCH-t: persistence α+β = {persistence:.3f}, long-run volatility ≈ {long_run:.1%}")
print(f"GJR asymmetry γ = {gjr.params['gamma[1]']:.3f} (t = {gjr.tvalues['gamma[1]']:.1f}): bad news raises volatility more")
pd.DataFrame({"GARCH-t": garch.params, "GJR-t": gjr.params})

## 3. Out-of-sample forecast competition

QLIKE loss: mean of $\;\text{RV}/\hat\sigma^2 - \ln(\text{RV}/\hat\sigma^2) - 1$ (lower is better).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def my_qlike(realized_var, forecast_var):
    ratio = np.asarray(realized_var) / np.asarray(forecast_var)
    return float(np.mean(ratio - np.log(ratio) - 1))

test_rv, test_fc = np.array([1.0, 2.0, 0.5]), np.array([1.2, 1.5, 0.7])
p.check("QLIKE", my_qlike(test_rv, test_fc), p.qlike(test_rv, test_fc));

In [ ]:
split = int(len(y) * 0.7)
fc = {}
for name, spec in {"GARCH-t": dict(o=0), "GJR-t": dict(o=1)}.items():
    res = arch_model(y, vol="GARCH", p=1, q=1, dist="t", **spec).fit(last_obs=y.index[split], disp="off")
    fc[name] = res.forecast(start=y.index[split], horizon=1, reindex=False).variance["h.1"].shift(1)   # forecast for the NEXT day
fc["EWMA"] = pd.Series(p.ewma_var(rets["SPY"].to_numpy()) * 1e4, index=y.index)
fc["21-day rolling"] = (y.rolling(21).var()).shift(1)
realized = y ** 2 + 1e-4                              # squared return as the (noisy) realized-variance proxy
table = {}
for name, f in fc.items():
    both = pd.concat([realized, f], axis=1).dropna()
    both = both.loc[both.index > y.index[split]]          # evaluate on the test period only
    table[name] = {"QLIKE": p.qlike(both.iloc[:, 0], both.iloc[:, 1]), "MSE": np.mean((both.iloc[:, 0] - both.iloc[:, 1]) ** 2)}
pd.DataFrame(table).T.sort_values("QLIKE")

## 4. Realized volatility from intraday data (synthetic 5-minute bars)

In [ ]:
i5 = p.intraday_5min("SPY")
rv = (i5["ret"] ** 2).groupby(i5.index.normalize()).sum()      # daily realized variance
rv_vol = np.sqrt(rv * 252)
ax = (rv_vol * 100).plot(title="Daily realized volatility from 5-minute returns (% annualized)"); ax.set_xlabel(""); plt.show()
print(f"Autocorrelation of daily realized variance: {pd.Series(rv.values).autocorr():.2f} (volatility is predictable)")

## Questions
1. Which model wins out of sample, and by how much? Would the ranking change with a different split?
2. Is there a leverage effect in TLT or GLD? (Change `rets["SPY"]` in section 2.)
3. Why is volatility so much easier to forecast than returns?